In [1]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
from tqdm import tqdm
import numpy as np

In [2]:
from netgen.stl import STLGeometry
from ngsolve import Mesh as NGSolveMesh

# The following code took hours of trial and error to get right
# DO NOT GET INTO TRYING TO CONVERT BETWEEN DIFFERENT MESH FORMATS!!!!!
# Time spent on this is time wasted:
# Wasted man hours 12.5
geo = STLGeometry("Bunny.stl") 
ngmesh = geo.GenerateMesh(maxh=8)

# Basicallay need to perform all selections in netgen mesh (ie boundary definitions)
for el in ngmesh.Elements2D():
    for v in el.vertices:
        x, y, z = ngmesh[v]
        # scale down mesh
        ngmesh[v][0] = x/50
        ngmesh[v][1] = y/50
        ngmesh[v][2] = z/50
        if z < 8:
            el.index = 1
            break
        if z > 109:
            el.index = 2
            break
        else:
            el.index = 3
            
ngmesh.SetBCName(0, "bottom")
ngmesh.SetBCName(2, "middle")
ngmesh.SetBCName(1, "top")

# Once as ngsolve mesh it is useless
mesh = NGSolveMesh(ngmesh)
mesh.GetBoundaries()

('bottom', 'top', 'middle')

In [3]:
festest = VectorH1(mesh, order=3)
gftest = GridFunction(festest, name="test")

In [4]:
gftest.Set(CF((1,1,1)), definedon=mesh.Boundaries("bottom"))
Draw(gftest, mesh, height="2vh")

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [5]:
gftest.Set(CF((1,1,1)), definedon=mesh.Boundaries("middle"))
Draw(gftest, mesh, height="2vh")

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [6]:
gftest.Set(CF((1,1,1)), definedon=mesh.Boundaries("top"))
Draw(gftest, mesh, height="2vh")

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [7]:
dt = 0.8
beta = .25 # 1/4 limit for unconditional stability, set higher for more damping
gamma = .5 # trapezoidal rule

eta = 1e-2 # viscoealstic damping CAREFUL:  i dont understand but cannot go above 1% for any fucking reason. else instability crashes ...

E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [8]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="bottom")
u,v = fes.TnT()
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

with TaskManager():
    m_int = u*v*dx
    m = BilinearForm(m_int)
    a_int = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx
    a = BilinearForm(a_int)
    c_int = 2 * InnerProduct( eta * Sym(Grad(u)), Sym(Grad(v))).Compile()*dx
    c = BilinearForm(c_int)
    mstar = BilinearForm(fes)
    mstar += m_int
    mstar += - gamma * dt * c_int
    mstar += beta * dt**2 * a_int
    pre = MultiGridPreconditioner(mstar, "bddc") # Adds factor 8 Speedup for some reason ...
    mstar.Assemble()
    a.Assemble()
    m.Assemble()
    
# Assembly seems to be well parallelized

In [9]:
omega = 1/12 # resoncance with  is around 1.5
force = - CF( (0,0,5))*cos(2*np.pi*omega*t) + CF( (0,5,0))*sin(2*np.pi*omega*t)  + CF( (0,0,6))*cos(2*np.pi*omega*t)
f = LinearForm(force*v*ds("top"))

In [10]:
from ngsolve.krylovspace import CGSolver

acc_inv = CGSolver(mstar.mat, pre, printrates=False, tol=1e-8)
# This is also quite fast

In [11]:
with TaskManager():
    gfa.Set(CF((0,0,0)))
    gfv.Set(CF((0,0,0)))
    gfu.Set(CF((0,0,0)))

In [12]:
gfut = GridFunction(gfu.space,multidim=0)

fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
gfstress = GridFunction(fesstress)
gfstresst = GridFunction(fesstress, multidim=0)

tend = t.Get()+48
steps = int((tend - t.Get()) / dt)

with TaskManager():
    for i in tqdm(range(steps), desc="Time-stepping", ncols=100, unit="step"):
        c.Assemble()

        f.Assemble()

        rhs_acceleration = f.vec - a.mat*(gfu.vec + dt*gfv.vec + .5*dt**2*(1-2*beta)*gfa.vec) + c.mat*(gfv.vec + (1-gamma)*gfa.vec) 
        
        new_acceleration = acc_inv * rhs_acceleration
        
        # Writing back into the grid functions is extremely slow and takes most of the iteration time
        # not paralellized
        gfu.vec.data = gfu.vec + dt*gfv.vec + 0.5*dt**2*((1-2*beta)*gfa.vec + 2*beta*new_acceleration)
        
        gfv.vec.data = gfv.vec + dt*((1-gamma)*gfa.vec + gamma*new_acceleration)
        
        gfa.vec.data = new_acceleration
        
        
        if i%4 == 0: # for some reason cannot take every step else the gui doesnt show
            gfut.AddMultiDimComponent(gfu.vec)
            
            gfstress.Interpolate(Stress(Sym(Grad(gfu))))
            
            gfstresst.AddMultiDimComponent(gfstress.vec)
        
        t.Set(t.Get()+dt)
    

Time-stepping: 100%|██████████████████████████████████████████████| 60/60 [17:21<00:00, 17.35s/step]


This is the issue with the Tasks:

![CPU Usage](btop_output.png)

One timestep loop in btop shows the CPU usage: First Inverting matrix is optimized for parallelization (large peak in utilization) but then whriting back to gfus takes a while and has bad utilization (5 smaller peaks for gfu, gfv, gfa, gut, gfstresst updates). 

Avoid using multiple GridFunctions in future!

In [13]:
# Should really be rendered on gpu but probably webgui does not allow
Draw (gfut, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [ ]:
Draw(gfstresst, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True);